In [9]:
import sys, os

# ── Locate the repo root robustly ────────────────────────────────────────────
# os.getcwd() in Jupyter/VS Code points to the notebook's directory.
REPO_ROOT = os.getcwd()
SOFT_DTW  = os.path.join(REPO_ROOT, "soft_dtw_balls")

# ── Clean up any path that would expose dtw_intersection.py as a flat module ─
# That happens when the dtw_intersection/ subfolder itself is on sys.path,
# because that folder contains dtw_intersection.py.
_bad = os.path.normpath(os.path.join(REPO_ROOT, "dtw_intersection"))
sys.path = [p for p in sys.path if os.path.normpath(p) != _bad]

# Wipe any stale cached import of dtw_intersection as a flat module
for _k in list(sys.modules):
    if _k == "dtw_intersection" or _k.startswith("dtw_intersection."):
        del sys.modules[_k]

# ── Add required roots (REPO_ROOT lets Python see dtw_intersection/ as package)
for p in [SOFT_DTW, REPO_ROOT]:
    if os.path.normpath(p) not in [os.path.normpath(x) for x in sys.path]:
        sys.path.insert(0, p)

print("REPO_ROOT :", REPO_ROOT)
print("dtw_intersection/ is a package :", os.path.isfile(os.path.join(REPO_ROOT, "dtw_intersection", "__init__.py")))
print("soft_dtw_balls/ exists          :", os.path.isdir(SOFT_DTW))

REPO_ROOT : /Users/janniskremer/test_master_dtw
dtw_intersection/ is a package : True
soft_dtw_balls/ exists          : True


In [10]:
import numpy as np
import matplotlib.pyplot as plt
import math, json, csv

# ── QP solver imports (Gurobi-optional) ─────────────────────────────────────
try:
    from dtw_intersection.dtw_intersection import dtw_exclusive_min_delta, HAS_GUROBI
    from dtw_intersection.shattering import test_dtw_shattering
    from dtw_intersection.shattering_capacity import estimate_shattering_capacity
    QP_AVAILABLE = HAS_GUROBI
    print("QP solver imported. Gurobi available:", HAS_GUROBI)
except ImportError as e:
    QP_AVAILABLE = False
    print(f"QP solver not available: {e}")

# ── Soft-DTW solver imports ──────────────────────────────────────────────────
try:
    from soft_dtw_solver import (
        SoftDTW,
        hard_dtw_distance,
        validate_witness_hard_dtw,
        optimize_ball_robust,
        check_shattering,
        sequential_capacity_estimation,
        load_point_set_from_csv,
        validate_csv_witnesses,
    )
    import torch
    SOFTDTW_AVAILABLE = True
    print("Soft-DTW solver imported. Torch version:", torch.__version__)
except ImportError as e:
    SOFTDTW_AVAILABLE = False
    print(f"Soft-DTW solver not available: {e}")

QP solver imported. Gurobi available: True
Soft-DTW solver imported. Torch version: 2.11.0


---
## Section 1 — Exact QP Solver (`dtw_intersection`)

The QP solver encodes the DTW dynamic programming recurrence as a
mixed-integer quadratically constrained program (MIQCP) and solves it with Gurobi.

### Example setup
We use **4 query curves of length 2** (code `k=2` = thesis `m=2`) and test whether they
can be shattered by DTW balls with **center length 3** (code `m=3` = thesis `k=3`).

This corresponds to thesis range space $\mathcal{R}^1_{k=3,\,m=2}$, for which the
theoretical lower bound is $k-1 = 2$.

In [11]:
# Example point set — 4 query curves of length 2 (code k=2 = thesis m=2)
# These are hand-chosen to be clearly separated in R^2.
query_curves_qp = [
    np.array([-3.0,  1.0]),   # Q0
    np.array([ 3.0,  1.0]),   # Q1
    np.array([-1.0, -3.0]),   # Q2
    np.array([ 1.0, -3.0]),   # Q3
]

# Code convention: m = center length = thesis k = 3
CENTER_LEN_QP = 3   # code m = thesis k
QUERY_LEN_QP  = 2   # code k = thesis m

print(f"Query curves: {len(query_curves_qp)} curves of length {QUERY_LEN_QP}")
print(f"Center length: {CENTER_LEN_QP}")
print(f"Range space: R^1_{{k={CENTER_LEN_QP}, m={QUERY_LEN_QP}}} (thesis notation)")
print(f"Subsets to test: 2^{len(query_curves_qp)} = {2**len(query_curves_qp)}")

Query curves: 4 curves of length 2
Center length: 3
Range space: R^1_{k=3, m=2} (thesis notation)
Subsets to test: 2^4 = 16


In [12]:
# 1a. Single subset separation
# Test: can we find a center P in R^3 that puts Q0, Q1 inside and Q2, Q3 outside?
#
# dtw_exclusive_min_delta(centers_inside, centers_outside, k, m)
#   - centers_inside / centers_outside : query curves of length k  (code k = thesis m)
#   - k : length of each query curve   (code k = thesis m)
#   - m : length of the witness center (code m = thesis k)
# Returns: (delta, witness) tuple, or (None, None) if infeasible.

if QP_AVAILABLE:
    inside_curves  = [query_curves_qp[i] for i in [0, 1]]
    outside_curves = [query_curves_qp[j] for j in [2, 3]]

    delta, witness = dtw_exclusive_min_delta(
        centers_inside  = inside_curves,
        centers_outside = outside_curves,
        k               = QUERY_LEN_QP,   # code k = thesis m = 2
        m               = CENTER_LEN_QP,  # code m = thesis k = 3
        epsilon         = 1e-4,
        verbose         = False,
    )

    feasible = delta is not None
    print(f"Subset I = [0, 1]")
    print(f"Feasible : {feasible}")
    if feasible:
        print(f"Delta    : {delta:.6g}")
        print(f"Witness P: {witness.round(4)}")
else:
    print("Gurobi not available — skipping QP separation test.")
    print("Install gurobipy and obtain a licence to run this cell.")


TypeError: dtw_exclusive_min_delta() missing 1 required positional argument: 'k'

In [ ]:
# ── 1b. Full shattering test (all 2^s subsets) ───────────────────────────────
#
# test_dtw_shattering(sequences, m)
#   - sequences : query curves (code k per curve = thesis m)
#   - m         : center length (code m = thesis k)

if QP_AVAILABLE:
    report = test_dtw_shattering(
        sequences = query_curves_qp,
        m         = CENTER_LEN_QP,   # code m = thesis k
        epsilon   = 1e-4,
        verbose   = True,
    )

    print(f"\n{'='*50}")
    print(f"Is shattered  : {report['is_shattered']}")
    print(f"Feasible      : {report['num_feasible']} / {report['num_subsets']}")
    if report['failed_subsets']:
        print("Failed subsets:", [sorted(s) for s in report['failed_subsets']])
else:
    print("⚠ Gurobi not available — skipping QP shattering test.")

In [ ]:
# 1c. Sequential capacity estimation
# Greedily builds the largest shatterable point set for thesis k=3, m=2.
#
# estimate_shattering_capacity(k, m, ...)
#   - k : code k = query length  = thesis m
#   - m : code m = center length = thesis k
# Returns dict; key 's_max' is the estimated shattering capacity.

if QP_AVAILABLE:
    cap_result = estimate_shattering_capacity(
        k        = QUERY_LEN_QP,    # code k = thesis m = 2
        m        = CENTER_LEN_QP,   # code m = thesis k = 3
        T        = 5,
        num_runs = 1,
        sampler  = "sphere-gaussian",
        max_s    = 8,
        verbose  = True,
    )

    d_max = cap_result["s_max"]
    print(f"\nSequential capacity estimate : s_max = {d_max}")
    print(f"(Thesis lower bound k-1 = {CENTER_LEN_QP - 1}; theory upper bound O(k log k))")
else:
    print("Gurobi not available — skipping QP capacity estimation.")


---
## Section 2 — Soft-DTW Gradient Descent (`soft_dtw_balls`)

The soft-DTW solver replaces the hard DTW minimum with the smooth
$\mathrm{dtw}^\gamma$ distance and optimises center $P$ and radius $\Delta$
jointly with Adam.

We use the same 4-point example as Section 1
(code `k=2`, `m=3`; thesis $k=3$, $m=2$), but also demonstrate the $k=m$ diagonal
which is the soft-DTW solver's main use case.

In [ ]:
# Same example, converted to torch tensors as expected by the soft-DTW API.
# Note: soft_dtw_solver uses code convention (k=query, m=center).

if SOFTDTW_AVAILABLE:
    query_curves_soft = [
        torch.tensor([-3.0,  1.0], dtype=torch.float32),  # Q0
        torch.tensor([ 3.0,  1.0], dtype=torch.float32),  # Q1
        torch.tensor([-1.0, -3.0], dtype=torch.float32),  # Q2
        torch.tensor([ 1.0, -3.0], dtype=torch.float32),  # Q3
    ]

    CENTER_LEN_SOFT = 3   # code m = thesis k
    GAMMA = 0.1           # soft-DTW smoothing; smaller → closer to hard DTW

    print(f"Query curves: {len(query_curves_soft)} × length {query_curves_soft[0].shape[0]}")
    print(f"Center length (code m = thesis k): {CENTER_LEN_SOFT}")
    print(f"Gamma: {GAMMA}")
else:
    print("⚠ soft_dtw_solver not available.")

In [ ]:
# ── 2a. Single subset: optimize_ball_robust ───────────────────────────────────
# Finds a center P that separates inside subset I from the outside.
#
# optimize_ball_robust(Qs, I, k, gamma)
#   - Qs    : all query curves
#   - I     : 0-based indices of the "inside" set
#   - k     : center length (code k in this function = code m in CSV convention)

if SOFTDTW_AVAILABLE:
    I_test = [0, 1]   # Q0, Q1 inside

    success, P_opt, Delta_opt, max_in, min_out, hard_valid = optimize_ball_robust(
        Qs      = query_curves_soft,
        I       = I_test,
        k       = CENTER_LEN_SOFT,  # center length
        gamma   = GAMMA,
        epochs  = 500,
        retries = 5,
        require_hard_dtw_validation = True,
        verbose = False,
    )

    print(f"Inside subset I = {I_test}")
    print(f"Success         : {success}")
    if success:
        print(f"Witness P       : {P_opt.numpy().round(4)}")
        print(f"Delta           : {float(Delta_opt):.6g}")
        print(f"max DTW inside  : {max_in:.6g}")
        print(f"min DTW outside : {min_out:.6g}")
        print(f"Gap (min_out - max_in): {min_out - max_in:.6g}")
        print(f"Hard-DTW valid  : {hard_valid}")
else:
    print("⚠ Soft-DTW solver not available.")

In [ ]:
# 2b. Full shattering check (all 2^s subsets)
#
# check_shattering(Qs, k, gamma, epochs, retries, validation, verbose)
#   Returns: (is_shattered: bool, witnesses: dict{tuple -> {P, Delta, max_in_dtw, min_out_dtw, hard_dtw_valid}})

if SOFTDTW_AVAILABLE:
    print(f"Testing shattering: {len(query_curves_soft)} points, "
          f"center length={CENTER_LEN_SOFT}, gamma={GAMMA}")
    print(f"Running {2**len(query_curves_soft)} subset optimisations...\n")

    is_shattered, witnesses = check_shattering(
        Qs         = query_curves_soft,
        k          = CENTER_LEN_SOFT,
        gamma      = GAMMA,
        epochs     = 300,
        retries    = 5,
        validation = True,
        verbose    = False,
    )

    print(f"Is shattered : {is_shattered}")
    print(f"Subsets found: {len(witnesses)} / {2**len(query_curves_soft)}")

    if witnesses:
        print("\nSample witnesses (subset -> Delta, gap):")
        for key, w in sorted(witnesses.items(), key=lambda t: len(t[0]))[:8]:
            gap = w['min_out_dtw'] - w['max_in_dtw']
            print(f"  I={str(list(key)):20s}  Delta={w['Delta']:.4f}  "
                  f"max_in={w['max_in_dtw']:.4f}  min_out={w['min_out_dtw']:.4f}  gap={gap:.4f}")
else:
    print("Soft-DTW solver not available.")


In [ ]:
# 2c. Sequential capacity estimation (k = m = 3 diagonal)
# Regime used for Figure 5.7 in the thesis.
# Code: k=3 (query), m=3 (center). Thesis: k=3, m=3.
#
# sequential_capacity_estimation(m, k, ...) — note m and k are separate args
#   num_runs=1 returns: (d_max, X_final, witnesses_final)

if SOFTDTW_AVAILABLE:
    KM = 3   # code k = code m = thesis k = thesis m = 3

    print(f"Sequential capacity estimation: code k=m={KM} (thesis k=m={KM})")
    print("This may take a minute...\n")

    d_max, X_final, witnesses_seq = sequential_capacity_estimation(
        m                   = KM,
        k                   = KM,
        gamma               = 0.1,
        max_retries_step4   = 5,
        epochs              = 300,
        retries             = 5,
        max_d               = 10,
        num_runs            = 1,
        validation          = True,
        sampling_mode       = "near_unit_sphere",
        witness_csv_path    = "/dev/null",  # suppress CSV output
    )

    print(f"\nCapacity estimate for k=m={KM}: d_max = {d_max}")
    print(f"(Thesis Figure 5.7 value: 6 for k=m=3)")
    if X_final:
        print(f"\nShattered point set ({len(X_final)} points):")
        for i, q in enumerate(X_final):
            arr = np.asarray(q.detach() if hasattr(q, 'detach') else q, dtype=float)
            print(f"  Q{i}: {arr.round(4)}")
else:
    print("Soft-DTW solver not available.")


---
## Section 3 — Witness Validation

This section verifies that witnesses stored in the experiment CSV files
**actually satisfy strict DTW separation** — i.e., that every stored
center $P$ and radius $\Delta$ achieves
$$\max_{i \in I} \mathrm{DTW}(P, Q_i) < \min_{j \notin I} \mathrm{DTW}(P, Q_j)$$
under hard (exact) DTW.

We use two CSV files:
- `experiments/fig5.4_qp_fixed_m2_thesis/k2_m7.csv` — the flagship 9-point witness
  (thesis $k=7$, $m=2$, code `k=2`, `m=7`)
- `experiments/fig5.7_softdtw_k_equals_m/witnesses_k3_m3_run1.csv` — soft-DTW witness
  for $k=m=3$

You can swap in any CSV from `experiments/` by changing `CSV_PATH` in the next cell.

In [ ]:
# ── 3a. Load and validate witnesses from a CSV file ───────────────────────────
# Change CSV_PATH to validate a different experiment file.

CSV_PATH = os.path.join(REPO_ROOT, "experiments",
                        "fig5.4_qp_fixed_m2_thesis", "k2_m7.csv")

print(f"Validating: {os.path.relpath(CSV_PATH, REPO_ROOT)}")
print()

if SOFTDTW_AVAILABLE and os.path.isfile(CSV_PATH):
    points, query_len, center_len, witnesses_csv = load_point_set_from_csv(CSV_PATH)

    print(f"Points loaded  : {len(points)}")
    print(f"Code k (query) = thesis m : {query_len}")
    print(f"Code m (center)= thesis k : {center_len}")
    print(f"Stored witnesses: {len(witnesses_csv)} subsets (expect 2^{len(points)} = {2**len(points)})")

    print("\n── Hard-DTW witness validation ─────────────────────────────────────")
    val_results = validate_csv_witnesses(points, witnesses_csv, verbose=True)

elif not os.path.isfile(CSV_PATH):
    print(f"⚠ File not found: {CSV_PATH}")
    print("  Run 'git submodule update --init --recursive' to fetch submodule data.")
else:
    print("⚠ Soft-DTW solver not available — cannot run validation.")

In [ ]:
# ── 3b. Visualise separation margins ─────────────────────────────────────────
# For each subset, plot the gap (min_out_DTW − max_in_DTW) as a bar chart.
# A gap > 0 confirms strict separation; gap ≤ 0 means the witness failed.

if SOFTDTW_AVAILABLE and 'val_results' in dir() and val_results:
    subsets = sorted(val_results.keys(), key=lambda t: (len(t), t))
    gaps    = [val_results[s]['min_out'] - val_results[s]['max_in'] for s in subsets]
    colors  = ['steelblue' if g > 0 else 'crimson' for g in gaps]
    labels  = ["{" + ",".join(str(i) for i in s) + "}" for s in subsets]

    fig, ax = plt.subplots(figsize=(max(8, len(subsets) * 0.35), 4))
    ax.bar(range(len(gaps)), gaps, color=colors, edgecolor='white', linewidth=0.3)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=6)
    ax.set_ylabel("Separation gap  $\\min_{j\\notin I}\\mathrm{DTW}(P,Q_j) - \\max_{i\\in I}\\mathrm{DTW}(P,Q_i)$")
    ax.set_title(f"Hard-DTW separation gaps — {os.path.basename(CSV_PATH)}\n"
                 f"Blue = valid, Red = failed  "
                 f"({sum(1 for g in gaps if g > 0)}/{len(gaps)} passed)")
    plt.tight_layout()
    plt.show()
else:
    print("No validation results to plot (run cell 3a first).")

In [ ]:
# ── 3c. Validate a custom witness manually ────────────────────────────────────
# Use this cell to check whether your own (P, Δ, point_set, subset I) constitutes
# a valid DTW-ball separation.
#
# Fill in:
#   CUSTOM_POINTS : list of query curves (code k per curve = thesis m)
#   CUSTOM_I      : 0-based indices of the inside set
#   CUSTOM_P      : candidate center (code m = thesis k)

# --- Edit below this line ---
CUSTOM_POINTS = [
    np.array([-3.0,  1.0]),   # Q0
    np.array([ 3.0,  1.0]),   # Q1
    np.array([-1.0, -3.0]),   # Q2
    np.array([ 1.0, -3.0]),   # Q3
]
CUSTOM_I = [0, 1]                          # inside set (0-based)
CUSTOM_P = np.array([0.0, 1.0, 0.0])      # candidate center (length = code m = thesis k)
# --- Stop editing ---

if SOFTDTW_AVAILABLE:
    is_valid, max_in, min_out, all_dtws = validate_witness_hard_dtw(
        P   = CUSTOM_P,
        Qs  = CUSTOM_POINTS,
        I   = CUSTOM_I,
    )

    not_I = [j for j in range(len(CUSTOM_POINTS)) if j not in CUSTOM_I]

    print(f"Point set     : {len(CUSTOM_POINTS)} curves")
    print(f"Inside  I     : {CUSTOM_I}")
    print(f"Outside ¬I    : {not_I}")
    print(f"Center P      : {CUSTOM_P.round(4)}")
    print()
    print("Per-point DTW² distances:")
    for idx, d in enumerate(all_dtws):
        role = "inside " if idx in CUSTOM_I else "outside"
        print(f"  Q{idx} ({role}): DTW²(P, Q{idx}) = {d:.6g}")
    print()
    print(f"max_in  : {max_in:.6g}")
    print(f"min_out : {min_out:.6g}")
    print(f"Gap     : {min_out - max_in:.6g}  ({'✓ strictly separated' if is_valid else '✗ NOT separated'})")
    print(f"\nWitness valid: {is_valid}")
else:
    print("⚠ Soft-DTW solver not available (needed for hard_dtw_distance).")

In [ ]:
# ── 3d. Validate the flagship k2_m7 witness from the thesis ──────────────────
# Load the 9-point shattered set (thesis k=7, m=2; code k=2, m=7).
# Prints a detailed per-subset report confirming all 2^9 = 512 witnesses.

K2M7_PATH = os.path.join(REPO_ROOT, "experiments",
                         "fig5.4_qp_fixed_m2_thesis", "k2_m7.csv")

if SOFTDTW_AVAILABLE and os.path.isfile(K2M7_PATH):
    pts, q_len, c_len, w_csv = load_point_set_from_csv(K2M7_PATH)

    print(f"k2_m7.csv  |  code k={q_len} (thesis m), code m={c_len} (thesis k)")
    print(f"Points: {len(pts)}, Witnesses: {len(w_csv)}, Expected: 2^{len(pts)} = {2**len(pts)}")

    val = validate_csv_witnesses(pts, w_csv, verbose=True)

    n_pass = sum(1 for r in val.values() if r['valid'])
    n_total = len(val)

    if n_pass == n_total:
        print(f"\n✅  All {n_total} witnesses are strictly separated under hard DTW.")
        print(f"    This certifies VCdim(R^1_{{k={c_len},m={q_len}}}) ≥ {len(pts)}  "
              f"(thesis notation: k={c_len}, m={q_len})")
    else:
        print(f"\n⚠  {n_total - n_pass} witnesses FAILED strict hard-DTW separation.")
elif not os.path.isfile(K2M7_PATH):
    print(f"⚠ File not found: {K2M7_PATH}")
else:
    print("⚠ Soft-DTW solver not available.")